# HyCAN-DB — Corpus overview

An exploratory overview of the HyCAN-DB hydrogen-sorption corpus: how many papers and samples it holds, which conditions and materials it covers, and the first-order structure of the uptake measurements. All numbers and figures are computed at runtime from `data/raw/measurements_v0.1.csv`; nothing here is hardcoded.

Figures are produced by the pure functions in `src/hycan/plotting.py` so the notebook stays a thin driver and the plotting logic is reusable and testable.

## 1. Setup

Import the scientific stack, report exact library versions for reproducibility, apply the shared house plotting style, and load the validated dataset. Version pinning and a single load point matter because every downstream number must trace back to one file and one environment.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

print(f'pandas     {pd.__version__}')
print(f'numpy      {np.__version__}')
print(f'matplotlib {matplotlib.__version__}')
print(f'seaborn    {sns.__version__}')


def find_repo_root(start=None):
    '''Walk up from `start` (cwd by default) to the dir holding the dataset.'''
    start = Path.cwd() if start is None else Path(start)
    for p in [start, *start.parents]:
        if (p / 'data' / 'raw' / 'measurements_v0.1.csv').exists():
            return p
    return None


REPO_ROOT = find_repo_root()
if REPO_ROOT is None:
    raise FileNotFoundError(
        'Could not locate repo root containing data/raw/measurements_v0.1.csv'
    )

SRC = REPO_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from hycan import plotting

plotting.set_house_style()

DATA_PATH = REPO_ROOT / 'data' / 'raw' / 'measurements_v0.1.csv'
FIGURES_DIR = REPO_ROOT / 'figures'
print('repo root:', REPO_ROOT)

In [ ]:
df = None
if not DATA_PATH.exists():
    print(f'ERROR: data file not found at {DATA_PATH}. Cannot proceed.')
elif DATA_PATH.stat().st_size == 0:
    print(f'ERROR: data file at {DATA_PATH} is empty. Cannot proceed.')
else:
    df = pd.read_csv(DATA_PATH)
    if df.empty:
        print(f'ERROR: data file at {DATA_PATH} has no rows. Cannot proceed.')
        df = None
    else:
        print(f'Loaded {DATA_PATH.name}')
        print('df.shape:', df.shape)

The version banner records the exact environment. If loading succeeded, `df.shape` above reports the row and column counts of the corpus as loaded; if the file were missing or empty, the cell would print an error and leave `df` unset rather than fabricating data.

## 2. Corpus summary

Headline counts that frame everything else: how many distinct papers and samples feed the corpus, how many measurement rows there are in total, the span of publication years, and which journals are represented. These establish the scale and provenance of the dataset at a glance.

In [ ]:
print('Unique papers (paper_id):  ', df['paper_id'].nunique())
print('Unique samples (sample_id):', df['sample_id'].nunique())
print('Total measurements (rows): ', len(df))
print('Year range:                ', int(df['year'].min()), '-', int(df['year'].max()))
print('Journals:', sorted(df['journal'].dropna().unique()))

The printout shows the corpus is built from a small number of papers and samples relative to its total measurement rows, spanning the reported year range and drawn from the listed journals. Multiple measurement rows per paper and per sample are expected, since each paper contributes several temperature/pressure points.

## 3. Material composition

A bar chart of how many measurement rows fall into each `material_class`. This shows where the corpus is measurement-rich versus thin across material families, which conditions later comparisons can and cannot be balanced over.

In [ ]:
counts = df['material_class'].value_counts()

fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(x=counts.index, y=counts.values, ax=ax, hue=counts.index, legend=False)
ax.set_title('Measurements per material class')
ax.set_xlabel('material_class')
ax.set_ylabel('number of measurements')
fig.tight_layout()
plt.show()

The bars show the relative number of measurement rows per material class. Taller bars mark the classes with the most rows in the corpus; shorter bars mark classes with only a handful of measurements.

## 4. Temperature and pressure coverage

Each measurement plotted in the temperature–pressure plane (pressure on a log axis), colored by material class. This reveals which regions of condition space the corpus actually samples and where it is sparse, using the shared `fig2_condition_space` function.

In [ ]:
fig2 = plotting.fig2_condition_space(df, FIGURES_DIR / 'fig2_condition_space.png')
plt.show()

The scatter shows which temperature–pressure combinations are populated. Points stack into vertical bands at the discrete temperatures used in the corpus, spread over a range of pressures on the log axis; colors indicate which material classes were measured at each condition.

## 5. Uptake distribution

Histograms of hydrogen uptake (wt%) split into two regimes — cryogenic (T ≤ 100 K) and room temperature (T ≥ 250 K) — with null uptakes dropped. Separating the regimes matters because cryogenic and near-ambient uptakes live on very different scales and should not be pooled into one distribution.

In [ ]:
cryo = df[df['temperature_k'] <= 100].dropna(subset=['uptake_wt_pct'])
room = df[df['temperature_k'] >= 250].dropna(subset=['uptake_wt_pct'])

pal = sns.color_palette('colorblind')
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
axes[0].hist(cryo['uptake_wt_pct'], bins=12, color=pal[0], edgecolor='white')
axes[0].set_title(f'Cryogenic (T <= 100 K), n={len(cryo)}')
axes[0].set_xlabel('uptake (wt%)')
axes[0].set_ylabel('number of measurements')
axes[1].hist(room['uptake_wt_pct'], bins=12, color=pal[1], edgecolor='white')
axes[1].set_title(f'Room temperature (T >= 250 K), n={len(room)}')
axes[1].set_xlabel('uptake (wt%)')
fig.tight_layout()
plt.show()

The two panels show the uptake distributions in each regime, with the sample count in each title. The cryogenic and room-temperature panels occupy different ranges of the uptake axis, reflecting the different scales of the two regimes.

## 6. Uptake vs BET surface area (Chahine rule)

For the 77 K subset only, uptake (wt%) against BET surface area, with the Chahine rule (`y = x / 500`, i.e. ~1 wt% per 500 m²/g) overlaid and points colored by material class. This checks how closely cryogenic uptake tracks surface area across material families, using the shared `fig3_chahine` function.

In [ ]:
fig3 = plotting.fig3_chahine(df, FIGURES_DIR / 'fig3_chahine.png')
plt.show()

The scatter shows the 77 K measurements (null-BET rows dropped) against the dashed Chahine reference line. Each point's position relative to the line indicates whether its uptake sits above, on, or below the ~1 wt% per 500 m²/g expectation; colors distinguish material classes.

## 7. Corpus map

The corpus as a publication timeline: a stacked bar of the number of papers per year, colored by material class (each paper counted once per year via deduplication on `paper_id`). Though labeled Figure 1 and placed last here, this is the orienting map of when and on what materials the literature was published, produced by `fig1_corpus_map`.

In [ ]:
fig1 = plotting.fig1_corpus_map(df, FIGURES_DIR / 'fig1_corpus_map.png')
plt.show()

The stacked bars show how many papers appear in each year of the corpus and, by color, which material classes those papers cover. Because papers are deduplicated on `paper_id` before counting, each bar segment reflects distinct papers rather than measurement rows.

## 8. Reproducibility tier breakdown

A bar chart of how many rows carry each `reproducibility_tier`. Tiers encode how confidently a measurement could be reconstructed from its source, so this shows the overall evidence quality profile of the corpus.

In [ ]:
tier_counts = df['reproducibility_tier'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(x=tier_counts.index, y=tier_counts.values, ax=ax, hue=tier_counts.index, legend=False)
ax.set_title('Rows per reproducibility tier')
ax.set_xlabel('reproducibility_tier')
ax.set_ylabel('number of rows')
fig.tight_layout()
plt.show()

The chart shows how many rows fall into each reproducibility tier present in the corpus. A single dominant bar indicates the rows are concentrated in one tier; multiple bars would indicate a spread of evidence quality.

## 9. Provisional observations

Three plain, factual observations drawn from the figures above — descriptive only, not interpretive claims about mechanism or generality:

- Most measurements in the corpus were taken at 77 K, visible as the densest vertical band in the temperature–pressure figure (Section 4).
- The reproducibility-tier chart (Section 8) concentrates the rows into a single tier rather than spreading them across several.
- In the Chahine figure (Section 6), the 77 K points are distributed around the dashed `y = x / 500` reference line rather than all lying exactly on it.